# Práctica 07 — Análisis gramatical y entidades

**Curso:** NLP Clásico con Python · **Clase 07** · **Tiempo estimado:** 60-75 minutos
**Notebook de clase:** `notebooks_clase/clase07_analisis_gramatical_entidades.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los ejercicios 1 a 5 no necesitan modelos: usan cifras, conjuntos y árboles escritos a mano y `spacy.blank`. El reto usa el corpus de `datos/` (en Colab, sube la carpeta `datos/`). Los textos son sintéticos y se escribieron para el curso (*ejemplo ilustrativo*).
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: `numpy`, `pandas` y `spacy`.

---
## Celda 0 — Setup y datos

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("spacy", "spacy")
print("Librerías listas.")

In [ ]:
import os
import re
import json
from collections import Counter

import numpy as np
import pandas as pd
import spacy
from spacy.tokens import Doc
from spacy.matcher import Matcher

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

MARCA = re.compile(r"\{([^{}|]+)\|(PER|ORG|LOC|MISC)\}")

def parsear(marcado):
    """Convierte 'Hola {Ana|PER}' en (texto, {(inicio, fin, etiqueta)})."""
    texto, ents, pos = "", set(), 0
    for m in MARCA.finditer(marcado):
        texto += marcado[pos:m.start()]
        ents.add((len(texto), len(texto) + len(m.group(1)), m.group(2)))
        texto += m.group(1)
        pos = m.end()
    return texto + marcado[pos:], ents

ETIQUETAS = ["PER", "ORG", "LOC", "MISC"]
print("Librerías cargadas correctamente.")

In [ ]:
# datos-es
# Cuatro frases de ejemplo: referencia (gold) y predicción de un modelo imaginario
EJEMPLO = [
    ("{Marta Vidal|PER} dirige la {Fundación Horizonte|ORG} en {Quito|LOC}.",
     "{Marta Vidal|PER} dirige la {Fundación|ORG} Horizonte en {Quito|LOC}."),
    ("Los {catalanes|MISC} visitaron {Lisboa|LOC}.",
     "Los catalanes visitaron {Lisboa|LOC}."),
    ("{Pedro Soler|PER} viajó a {Toledo|LOC}.",
     "{Pedro Soler|PER} viajó a {Toledo|ORG}."),
    ("Hoy hace buen tiempo.",
     "{Hoy|PER} hace buen tiempo."),
]
GOLD_EJ, PRED_EJ = [], []
for _ref, _pred in EJEMPLO:
    _t1, _g = parsear(_ref)
    _t2, _p = parsear(_pred)
    assert _t1 == _t2
    GOLD_EJ.append(_g); PRED_EJ.append(_p)

# Dos árboles escritos a mano (cabezas absolutas): español e italiano
def _arbol(idioma, palabras, espacios, heads, deps, pos, lemmas):
    return Doc(spacy.blank(idioma).vocab, words=palabras, spaces=espacios, heads=heads, deps=deps, pos=pos, lemmas=lemmas)

DOC_ES = _arbol("es", ["La", "alcaldesa", "inauguró", "la", "feria", "y", "el", "concejal", "visitó", "el", "museo", "."],
                [True] * 11 + [False],
                [1, 2, 2, 4, 2, 8, 7, 8, 2, 10, 8, 2],
                ["det", "nsubj", "ROOT", "det", "obj", "cc", "det", "nsubj", "conj", "det", "obj", "punct"],
                ["DET", "NOUN", "VERB", "DET", "NOUN", "CCONJ", "DET", "NOUN", "VERB", "DET", "NOUN", "PUNCT"],
                ["el", "alcaldesa", "inaugurar", "el", "feria", "y", "el", "concejal", "visitar", "el", "museo", "."])
DOC_IT = _arbol("it", ["Il", "sindaco", "ha", "inaugurato", "la", "fiera", "e", "l'", "assessore", "ha", "visitato", "il", "museo", "."],
                [True, True, True, True, True, True, True, False, True, True, True, True, False, False],
                [1, 3, 3, 3, 5, 3, 10, 8, 10, 10, 3, 12, 10, 3],
                ["det", "nsubj", "aux", "ROOT", "det", "obj", "cc", "det", "nsubj", "aux", "conj", "det", "obj", "punct"],
                ["DET", "NOUN", "AUX", "VERB", "DET", "NOUN", "CCONJ", "DET", "NOUN", "AUX", "VERB", "DET", "NOUN", "PUNCT"],
                ["il", "sindaco", "avere", "inaugurare", "il", "fiera", "e", "il", "assessore", "avere", "visitare", "il", "museo", "."])

## Ejercicio 1 — Precisión, recall y F1

Completa `prf(tp, fp, fn)`: debe devolver la tupla `(precisión, recall, F1)`. Si un denominador es 0, el valor correspondiente es `0.0` (y F1 es `0.0` si precisión + recall es 0).

Antes de programar, calcula a mano con `tp = 4`, `fp = 3`, `fn = 3` (las cifras de la clase): P, R y F1 valen 4/7.

In [ ]:
def prf(tp, fp, fn):
    precision = None      # TODO: tp / (tp + fp), o 0.0 si el denominador es 0
    recall = None         # TODO: tp / (tp + fn), o 0.0 si el denominador es 0
    f1 = None             # TODO: media armónica de las dos
    return precision, recall, f1

# Comprobación automática
_r = prf(4, 3, 3)
assert None not in _r, "Completa las líneas marcadas con TODO"
assert np.allclose(_r, (4 / 7, 4 / 7, 4 / 7)), "con tp=4, fp=3, fn=3 los tres valores son 4/7"
assert np.allclose(prf(3, 1, 0), (0.75, 1.0, 6 / 7)), "con tp=3, fp=1, fn=0: P=0,75, R=1 y F1=6/7"
assert prf(0, 0, 1) == (0.0, 0.0, 0.0) and prf(0, 0, 0) == (0.0, 0.0, 0.0), "cuidado con la división por cero"
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Contar TP, FP y FN por etiqueta

Completa `contar_exacto(gold, pred)`. `gold` y `pred` son listas (una posición por frase) de conjuntos de entidades `(inicio, fin, etiqueta)`. Devuelve un diccionario `{etiqueta: (tp, fp, fn)}` con las cuatro etiquetas de `ETIQUETAS`, en modo **exacto** (tramo y etiqueta iguales).

Pista: por frase, los TP son `g & p`, los FP son `p - g` y los FN son `g - p`. Se comprueba con `GOLD_EJ` y `PRED_EJ`: cuenta antes a mano el total (4 TP, 3 FP, 3 FN).

In [ ]:
def contar_exacto(gold, pred):
    cuenta = {e: [0, 0, 0] for e in ETIQUETAS}
    for g, p in zip(gold, pred):
        pass    # TODO: suma en cuenta[etiqueta][0], [1] o [2] los elementos de g & p, p - g y g - p
    return None      # TODO: devuelve {etiqueta: tuple(recuento)}

cuenta_ej = contar_exacto(GOLD_EJ, PRED_EJ)

# Comprobación automática
assert cuenta_ej is not None, "Completa las líneas marcadas con TODO"
assert cuenta_ej == {"PER": (2, 1, 0), "ORG": (0, 2, 1), "LOC": (2, 0, 1), "MISC": (0, 0, 1)}, "revisa el recuento por etiqueta (un error de etiqueta es FP de una etiqueta y FN de otra)"
assert tuple(sum(v[k] for v in cuenta_ej.values()) for k in range(3)) == (4, 3, 3), "el total debe ser 4 TP, 3 FP y 3 FN"
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Tipos de error

Completa `tipo_error(g, predichas)`. `g` es una entidad de la referencia `(inicio, fin, etiqueta)` que **no** fue acertada, y `predichas` es el conjunto de entidades predichas en esa frase. Devuelve:

- `"etiqueta"` si alguna predicha tiene el **mismo tramo** (con otra etiqueta),
- `"frontera"` si alguna predicha se **solapa** con `g` pero el tramo no coincide,
- `"omitida"` si ninguna predicha se solapa con `g`.

Dos tramos `(a1, b1)` y `(a2, b2)` se solapan si `a1 < b2 and a2 < b1`.

In [ ]:
def tipo_error(g, predichas):
    # TODO: comprueba primero el mismo tramo, después el solapamiento
    return None

# Comprobación automática
_tipos = [tipo_error(g, p) for gs, p in zip(GOLD_EJ, PRED_EJ) for g in gs - p]
assert None not in _tipos, "Completa las líneas marcadas con TODO"
assert sorted(_tipos) == ["etiqueta", "frontera", "omitida"], "la primera entidad es de frontera, la segunda omitida y la tercera de etiqueta"
assert tipo_error((0, 5, "PER"), {(0, 5, "LOC")}) == "etiqueta" and tipo_error((0, 5, "PER"), {(3, 9, "PER")}) == "frontera" and tipo_error((0, 5, "PER"), {(5, 9, "PER")}) == "omitida", "los tramos (0, 5) y (5, 9) no se solapan"
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Sujeto, verbo y objeto desde el árbol

`DOC_ES` y `DOC_IT` son árboles escritos a mano (no hace falta ningún modelo). Completa `extraer_svo(doc)`: para cada token con `pos_ == "VERB"`, busca entre sus hijos (`token.children`) uno con `dep_` que empiece por `nsubj` y otro con `dep_ == "obj"`; si están los dos, añade la terna `(texto del subárbol del sujeto, lema del verbo, texto del subárbol del objeto)`.

Para obtener el texto de un subárbol usa `"".join(x.text_with_ws for x in token.subtree).strip()`. El auxiliar italiano (`ha`) no es `VERB`: no lo tengas en cuenta.

In [ ]:
def extraer_svo(doc):
    ternas = []
    for t in doc:
        pass    # TODO: si t es VERB y tiene hijo nsubj y hijo obj, añade (sujeto, t.lemma_, objeto)
    return None      # TODO: devuelve la lista de ternas

svo_es = extraer_svo(DOC_ES)
svo_it = extraer_svo(DOC_IT)

# Comprobación automática
assert svo_es is not None and svo_it is not None, "Completa las líneas marcadas con TODO"
assert svo_es == [("La alcaldesa", "inaugurar", "la feria"), ("el concejal", "visitar", "el museo")], "el árbol español tiene dos ternas"   # dato-es
assert svo_it == [("Il sindaco", "inaugurare", "la fiera"), ("l'assessore", "visitare", "il museo")], "el árbol italiano tiene dos ternas; el auxiliar no cuenta"   # dato-es
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Un `EntityRuler` y un `Matcher`

Sin modelos: usa `spacy.blank(idioma)`, que solo tokeniza.

- Completa `crear_nlp_reglas(idioma, siglas, gentilicios)`: devuelve una *pipeline* en blanco con un `entity_ruler` que etiqueta cada sigla como `ORG` (patrón `[{"ORTH": sigla}]`) y cada gentilicio como `MISC` (patrón `[{"LOWER": {"IN": gentilicios}}]`).
- Completa `buscar_titulos(nlp, texto)`: con un `Matcher`, devuelve los textos de las secuencias de una o más palabras con mayúscula inicial (`{"IS_TITLE": True, "OP": "+"}`) que **siguen** a la palabra `de`, quedándote con las secuencias más largas (`spacy.util.filter_spans`).

In [ ]:
def crear_nlp_reglas(idioma, siglas, gentilicios):
    nlp = spacy.blank(idioma)
    ruler = nlp.add_pipe("entity_ruler")
    patrones = None      # TODO: lista de {"label": ..., "pattern": [...]}: una por sigla y una para los gentilicios
    ruler.add_patterns(patrones)
    return nlp

def buscar_titulos(nlp, texto):
    matcher = Matcher(nlp.vocab)
    # TODO: añade el patrón [{"LOWER": "de"}, {"IS_TITLE": True, "OP": "+"}] y filtra con filter_spans; recuerda quitar la palabra "de"
    return None

nlp_es = crear_nlp_reglas("es", ["ONU", "UE"], ["italianos", "catalanes"])      # dato-es
ents_es = [(e.text, e.label_) for e in nlp_es("La ONU y los italianos hablaron con la UE.").ents]      # dato-es
titulos = buscar_titulos(spacy.blank("es"), "La Universidad de Salamanca y el Museo de Historia Natural.")      # dato-es

# Comprobación automática
assert titulos is not None, "Completa las líneas marcadas con TODO"
assert ents_es == [("ONU", "ORG"), ("italianos", "MISC"), ("UE", "ORG")], "el ruler debe etiquetar ONU y UE como ORG e italianos como MISC"
assert titulos == ["Salamanca", "Historia Natural"], "el Matcher debe devolver las secuencias con mayúscula tras «de», sin la palabra «de»"   # dato-es
print("Ejercicio 5 correcto ✔")

---
## Reto final — Evaluar un sistema de reglas sobre el corpus real

Con `datos/noticias_ner_es.csv` y `datos/noticias_ner_it.csv` (referencia anotada a mano):

1. Escribe `evaluar_corpus(idioma, nlp)`: lee el corpus de ese idioma (`"es"` o `"it"`), procesa cada texto con `nlp` y devuelve `(precision, recall, f1)` micro en modo exacto, usando tus funciones `contar_exacto` y `prf` de los ejercicios 1 y 2. Devuelve también el número de entidades de la referencia.
2. Evalúa una *pipeline* **solo de reglas** (`crear_nlp_reglas` con las siglas `ONU`, `OMS`, `FMI`, `UE`, `UNESCO`, `UGT` y `CGIL` y sin gentilicios) en cada idioma.

Piensa qué esperas: sin modelo estadístico, ¿cuál debería ser la precisión? ¿Y el *recall*? Se comprueba con un cálculo independiente sobre la referencia.

> **Nota:** el corpus es pequeño y sintético (*ejemplo ilustrativo*); las cifras no se generalizan.

In [ ]:
SIGLAS_RETO = ["ONU", "OMS", "FMI", "UE", "UNESCO", "UGT", "CGIL"]

def evaluar_corpus(idioma, nlp):
    df = pd.read_csv(cargar_datos(f"noticias_ner_{idioma}.csv"))
    gold = [{(a, b, e) for a, b, e in json.loads(x)} for x in df["entidades"]]
    # TODO: predice con nlp.pipe(df["texto"]) y calcula (P, R, F1) con contar_exacto y prf
    return None, sum(len(g) for g in gold)

resultado = {i: evaluar_corpus(i, crear_nlp_reglas(i, SIGLAS_RETO, [])) for i in ("es", "it")}      # dato-es

# Comprobación automática: cálculo independiente sobre la referencia
assert all(v[0] is not None for v in resultado.values()), "Completa las líneas marcadas con TODO"
for idioma, ((p, r, f), n_gold) in resultado.items():
    df = pd.read_csv(cargar_datos(f"noticias_ner_{idioma}.csv"))
    aciertos = sum(1 for t, x in zip(df["texto"], df["entidades"]) for a, b, e in json.loads(x) if e == "ORG" and t[a:b] in SIGLAS_RETO)
    assert aciertos > 5 and n_gold >= 100
    assert np.isclose(p, 1.0), "todas las siglas de la lista son ORG en la referencia: precisión 1"
    assert np.isclose(r, aciertos / n_gold), "el recall es aciertos / entidades de la referencia"
    assert np.isclose(f, 2 * p * r / (p + r))
print("Reto final correcto ✔")
print({i: tuple(round(x, 3) for x in v[0]) for i, v in resultado.items()})

**Tus conclusiones (escribe 3-4 frases):** una *pipeline* solo de reglas tiene precisión perfecta y *recall* muy bajo. ¿Por qué? ¿En qué se diferencia de lo que ocurre con un modelo estadístico? ¿Cómo combinarías reglas y modelo y con qué frases medirías si la combinación mejora?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def prf(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

# Comprobación automática
_r = prf(4, 3, 3)
assert None not in _r, "Completa las líneas marcadas con TODO"
assert np.allclose(_r, (4 / 7, 4 / 7, 4 / 7)), "con tp=4, fp=3, fn=3 los tres valores son 4/7"
assert np.allclose(prf(3, 1, 0), (0.75, 1.0, 6 / 7)), "con tp=3, fp=1, fn=0: P=0,75, R=1 y F1=6/7"
assert prf(0, 0, 1) == (0.0, 0.0, 0.0) and prf(0, 0, 0) == (0.0, 0.0, 0.0), "cuidado con la división por cero"
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
def contar_exacto(gold, pred):
    cuenta = {e: [0, 0, 0] for e in ETIQUETAS}
    for g, p in zip(gold, pred):
        for _, _, e in g & p:
            cuenta[e][0] += 1
        for _, _, e in p - g:
            cuenta[e][1] += 1
        for _, _, e in g - p:
            cuenta[e][2] += 1
    return {e: tuple(v) for e, v in cuenta.items()}

cuenta_ej = contar_exacto(GOLD_EJ, PRED_EJ)

# Comprobación automática
assert cuenta_ej is not None, "Completa las líneas marcadas con TODO"
assert cuenta_ej == {"PER": (2, 1, 0), "ORG": (0, 2, 1), "LOC": (2, 0, 1), "MISC": (0, 0, 1)}, "revisa el recuento por etiqueta (un error de etiqueta es FP de una etiqueta y FN de otra)"
assert tuple(sum(v[k] for v in cuenta_ej.values()) for k in range(3)) == (4, 3, 3), "el total debe ser 4 TP, 3 FP y 3 FN"
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
def tipo_error(g, predichas):
    if any(g[:2] == p[:2] for p in predichas):
        return "etiqueta"
    if any(g[0] < p[1] and p[0] < g[1] for p in predichas):
        return "frontera"
    return "omitida"

# Comprobación automática
_tipos = [tipo_error(g, p) for gs, p in zip(GOLD_EJ, PRED_EJ) for g in gs - p]
assert None not in _tipos, "Completa las líneas marcadas con TODO"
assert sorted(_tipos) == ["etiqueta", "frontera", "omitida"], "la primera entidad es de frontera, la segunda omitida y la tercera de etiqueta"
assert tipo_error((0, 5, "PER"), {(0, 5, "LOC")}) == "etiqueta" and tipo_error((0, 5, "PER"), {(3, 9, "PER")}) == "frontera" and tipo_error((0, 5, "PER"), {(5, 9, "PER")}) == "omitida", "los tramos (0, 5) y (5, 9) no se solapan"
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def extraer_svo(doc):
    ternas = []
    for t in doc:
        if t.pos_ != "VERB":
            continue
        sujetos = [h for h in t.children if h.dep_.startswith("nsubj")]
        objetos = [h for h in t.children if h.dep_ == "obj"]
        if sujetos and objetos:
            sujeto = "".join(x.text_with_ws for x in sujetos[0].subtree).strip()
            objeto = "".join(x.text_with_ws for x in objetos[0].subtree).strip()
            ternas.append((sujeto, t.lemma_, objeto))
    return ternas

svo_es = extraer_svo(DOC_ES)
svo_it = extraer_svo(DOC_IT)

# Comprobación automática
assert svo_es is not None and svo_it is not None, "Completa las líneas marcadas con TODO"
assert svo_es == [("La alcaldesa", "inaugurar", "la feria"), ("el concejal", "visitar", "el museo")], "el árbol español tiene dos ternas"   # dato-es
assert svo_it == [("Il sindaco", "inaugurare", "la fiera"), ("l'assessore", "visitare", "il museo")], "el árbol italiano tiene dos ternas; el auxiliar no cuenta"   # dato-es
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
def crear_nlp_reglas(idioma, siglas, gentilicios):
    nlp = spacy.blank(idioma)
    ruler = nlp.add_pipe("entity_ruler")
    patrones = [{"label": "ORG", "pattern": [{"ORTH": s}]} for s in siglas]
    patrones.append({"label": "MISC", "pattern": [{"LOWER": {"IN": gentilicios}}]})
    ruler.add_patterns(patrones)
    return nlp

def buscar_titulos(nlp, texto):
    from spacy.util import filter_spans
    matcher = Matcher(nlp.vocab)
    matcher.add("DE_TITULO", [[{"LOWER": "de"}, {"IS_TITLE": True, "OP": "+"}]])
    doc = nlp(texto)
    tramos = filter_spans([doc[a:b] for _, a, b in matcher(doc)])
    return [t[1:].text for t in tramos]

nlp_es = crear_nlp_reglas("es", ["ONU", "UE"], ["italianos", "catalanes"])      # dato-es
ents_es = [(e.text, e.label_) for e in nlp_es("La ONU y los italianos hablaron con la UE.").ents]      # dato-es
titulos = buscar_titulos(spacy.blank("es"), "La Universidad de Salamanca y el Museo de Historia Natural.")      # dato-es

# Comprobación automática
assert titulos is not None, "Completa las líneas marcadas con TODO"
assert ents_es == [("ONU", "ORG"), ("italianos", "MISC"), ("UE", "ORG")], "el ruler debe etiquetar ONU y UE como ORG e italianos como MISC"
assert titulos == ["Salamanca", "Historia Natural"], "el Matcher debe devolver las secuencias con mayúscula tras «de», sin la palabra «de»"   # dato-es
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
SIGLAS_RETO = ["ONU", "OMS", "FMI", "UE", "UNESCO", "UGT", "CGIL"]

def evaluar_corpus(idioma, nlp):
    df = pd.read_csv(cargar_datos(f"noticias_ner_{idioma}.csv"))
    gold = [{(a, b, e) for a, b, e in json.loads(x)} for x in df["entidades"]]
    pred = [{(e.start_char, e.end_char, e.label_) for e in doc.ents} for doc in nlp.pipe(df["texto"])]
    cuenta = contar_exacto(gold, pred)
    tp, fp, fn = (sum(v[k] for v in cuenta.values()) for k in range(3))
    return prf(tp, fp, fn), sum(len(g) for g in gold)

resultado = {i: evaluar_corpus(i, crear_nlp_reglas(i, SIGLAS_RETO, [])) for i in ("es", "it")}      # dato-es

assert all(v[0] is not None for v in resultado.values()), "Completa las líneas marcadas con TODO"
for idioma, ((p, r, f), n_gold) in resultado.items():
    df = pd.read_csv(cargar_datos(f"noticias_ner_{idioma}.csv"))
    aciertos = sum(1 for t, x in zip(df["texto"], df["entidades"]) for a, b, e in json.loads(x) if e == "ORG" and t[a:b] in SIGLAS_RETO)
    assert aciertos > 5 and n_gold >= 100
    assert np.isclose(p, 1.0), "todas las siglas de la lista son ORG en la referencia: precisión 1"
    assert np.isclose(r, aciertos / n_gold), "el recall es aciertos / entidades de la referencia"
    assert np.isclose(f, 2 * p * r / (p + r))
print("Reto final correcto ✔")
print({i: tuple(round(x, 3) for x in v[0]) for i, v in resultado.items()})

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto no te ha quedado claro (POS frente a dependencias, acierto exacto frente a parcial, precisión frente a *recall*, etiqueta frente a frontera, `Matcher` frente a `EntityRuler`)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*